# 22. Three-Class Model (BUY / HOLD / SELL)

**Цель:** обучить модель с тремя классами (BUY, HOLD, SELL) и сравнить с продовым champion (CatBoost 2-class, prod_hold 25-70) на честном бэктесте (полные данные BUY+SELL+HOLD).

**План:**
1. Загрузка данных как в NB 16 (полные данные с HOLD)
2. Маппинг: -1→SELL(0), 0→HOLD(1), 1→BUY(2)
3. Rolling-фичи на полных данных, temporal split 7/1/2 (как NB16)
4. LightGBM multiclass
5. Бэктест signal_flip (argmax → BUY/SELL/HOLD), комиссия: разворот long↔short = 2 комиссии
6. Сравнение 2-class (prod_cat_seq, 25-70) vs 3-class

## 1. Импорты и константы

In [1]:
import sys, os, numpy as np, pandas as pd, joblib, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, accuracy_score
import lightgbm as lgb

warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

COMMISSION_RT = 0.001
SEQ_WINDOWS = [5, 15, 30, 60]
TARGET_COL = 'target'
MODEL_3CLASS_PATH = os.path.join(_root, 'models', 'prod_lgbm_3class.joblib')
MODEL_2CLASS_PATH = os.path.join(_root, 'models', 'prod_cat_seq.joblib')

print(f'Root: {_root}')
print(f'3-class model save: {MODEL_3CLASS_PATH}')
print(f'2-class model (for comparison): {MODEL_2CLASS_PATH}')

Root: c:\project\trading_bot_2Engine
3-class model save: c:\project\trading_bot_2Engine\models\prod_lgbm_3class.joblib
2-class model (for comparison): c:\project\trading_bot_2Engine\models\prod_cat_seq.joblib


## 2. Загрузка данных и маппинг 3-классов

Полные данные (BUY+SELL+HOLD). Маппинг для multiclass: -1→0 (SELL), 0→1 (HOLD), 1→2 (BUY).

In [2]:
labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')

df_raw = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df_raw.columns]

# Полные данные: BUY + SELL + HOLD
valid = df_raw.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 0.0, 1.0])]
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date
# Маппинг: -1→0 (SELL), 0→1 (HOLD), 1→2 (BUY)
valid['y'] = valid[TARGET_COL].map({-1.0: 0, 0.0: 1, 1.0: 2}).astype(int)
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = valid.dropna(subset=['ret_next'])

KEY_FEATS = BASE_FEATURES[:10]
grp = valid.groupby('session_key', group_keys=False)
for w in SEQ_WINDOWS:
    for c in KEY_FEATS:
        valid[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
        valid[f'{c}_roll{w}_std'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

SEQ_FEATURES = [f'{c}_roll{w}_{s}' for w in SEQ_WINDOWS for c in KEY_FEATS for s in ('mean', 'std')]
ALL_FEATURES = BASE_FEATURES + SEQ_FEATURES
valid = valid.dropna(subset=ALL_FEATURES)

dates = sorted(valid['date'].unique())
assert len(dates) >= 10, f'Нужно >= 10 дней, найдено {len(dates)}'
train_dates = set(dates[:7])
val_dates = set([dates[7]])
test_dates = set(dates[8:10])

sort_col = 'datetime' if 'datetime' in valid.columns else 'timestamp'
train_df = valid[valid['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
val_df = valid[valid['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df = valid[valid['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)

print(f'Dates: train={min(train_dates)}..{max(train_dates)} | val={dates[7]} | test={dates[8]}, {dates[9]}')
print(f'Rows: train={len(train_df):,} | val={len(val_df):,} | test={len(test_df):,}')
print(f'Target dist train: {dict(train_df["y"].value_counts().sort_index())}')
print(f'Features: {len(ALL_FEATURES)} ({len(BASE_FEATURES)} base + {len(SEQ_FEATURES)} rolling)')

Dates: train=2026-02-01..2026-02-07 | val=2026-02-08 | test=2026-02-09, 2026-02-10
Rows: train=212,858 | val=45,592 | test=74,437
Target dist train: {0: np.int64(62046), 1: np.int64(88887), 2: np.int64(61925)}
Features: 101 (21 base + 80 rolling)


## 3. Масштабирование и обучение LightGBM multiclass

In [3]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[ALL_FEATURES].fillna(0))
X_val = scaler.transform(val_df[ALL_FEATURES].fillna(0))
X_test = scaler.transform(test_df[ALL_FEATURES].fillna(0))
y_train = train_df['y'].values
y_val = val_df['y'].values
y_test = test_df['y'].values

model_3c = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)

model_3c.fit(X_train, y_train)

pred_val = model_3c.predict(X_val)
pred_test = model_3c.predict(X_test)
acc_val = accuracy_score(y_val, pred_val)
acc_test = accuracy_score(y_test, pred_test)

print(f'Accuracy val:  {acc_val:.6f}')
print(f'Accuracy test: {acc_test:.6f}')
print('\n=== Classification Report (test) ===')
print(classification_report(y_test, pred_test, target_names=['SELL(0)', 'HOLD(1)', 'BUY(2)'], digits=4))

Accuracy val:  0.552772
Accuracy test: 0.523785

=== Classification Report (test) ===
              precision    recall  f1-score   support

     SELL(0)     0.4684    0.3183    0.3790     18779
     HOLD(1)     0.5540    0.7597    0.6408     34962
      BUY(2)     0.4696    0.3118    0.3747     20696

    accuracy                         0.5238     74437
   macro avg     0.4974    0.4632    0.4648     74437
weighted avg     0.5090    0.5238    0.5008     74437



## 4. Бэктест signal_flip для 3-class

argmax(proba) → 0=SELL, 1=HOLD, 2=BUY. signal_flip: BUY=+1, SELL=-1, HOLD сохраняет предыдущую позицию.

**Комиссия:** при развороте long↔short — 2 комиссии (0.1% round-trip); при flat↔pos — 1 комиссия (0.05%).

In [4]:
def _fee_total(pos, pos_prev, pos_changed, commission_rt):
    """Разворот long↔short = 2 комиссии (commission_rt), flat↔pos = 1 комиссия (commission_rt/2)."""
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_per = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0)
    return fee_per.sum()


def backtest_3class(proba_3c, ret, session_ids, commission_rt=COMMISSION_RT):
    """proba_3c shape (n, 3): [P(SELL), P(HOLD), P(BUY)]. argmax → 0/1/2."""
    pred = np.argmax(proba_3c, axis=1)  # 0=SELL, 1=HOLD, 2=BUY
    n = len(pred)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 2:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    if session_ids is not None:
        sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    fee_total_val = _fee_total(pos, pos_prev, pos_changed, commission_rt)
    pnl_net = (pos * ret).sum() - fee_total_val
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_trade}


def backtest_prod_hold_asym(proba, ret, session_ids, thr_hi=0.70, thr_lo=0.25, commission_rt=COMMISSION_RT):
    """prod_hold 25-70: BUY >= thr_hi, SELL <= thr_lo, HOLD keeps previous. Разворот = 2 комиссии."""
    pred = np.where(proba >= thr_hi, 1, np.where(proba <= thr_lo, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    if session_ids is not None:
        sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    fee_total_val = _fee_total(pos, pos_prev, pos_changed, commission_rt)
    pnl_net = (pos * ret).sum() - fee_total_val
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_trade}


proba_val_3c = model_3c.predict_proba(X_val)
proba_test_3c = model_3c.predict_proba(X_test)

bt_val_3c = backtest_3class(proba_val_3c, val_df['ret_next'].values, val_df['session_key'].values)
bt_test_3c = backtest_3class(proba_test_3c, test_df['ret_next'].values, test_df['session_key'].values)

print('=== 3-Class Backtest (full data) ===')
print(f'VAL:  net={bt_val_3c["net_%"]:+.2f}%  trades={bt_val_3c["trades"]}  avg/trade={bt_val_3c["avg_%_per_trade"]:.4f}%')
print(f'TEST: net={bt_test_3c["net_%"]:+.2f}%  trades={bt_test_3c["trades"]}  avg/trade={bt_test_3c["avg_%_per_trade"]:.4f}%')

=== 3-Class Backtest (full data) ===
VAL:  net=+2381.79%  trades=4401  avg/trade=0.5412%
TEST: net=+2805.27%  trades=7918  avg/trade=0.3543%


## 5. Сравнение 2-class (champion) vs 3-class

Загружаем продовую 2-class модель (prod_cat_seq, 25-70) и считаем бэктест на тех же val/test (полные данные).

In [5]:
bundle_2c = joblib.load(MODEL_2CLASS_PATH)
model_2c = bundle_2c['model']
scaler_2c = bundle_2c['scaler']
thr_hi = bundle_2c.get('threshold', 0.70)
thr_lo = bundle_2c.get('threshold_lo', 0.25)

feats_2c = bundle_2c['features']
proba_val_2c = model_2c.predict_proba(scaler_2c.transform(val_df[feats_2c].fillna(0)))[:, 1]
proba_test_2c = model_2c.predict_proba(scaler_2c.transform(test_df[feats_2c].fillna(0)))[:, 1]

bt_val_2c = backtest_prod_hold_asym(proba_val_2c, val_df['ret_next'].values, val_df['session_key'].values, thr_hi=thr_hi, thr_lo=thr_lo)
bt_test_2c = backtest_prod_hold_asym(proba_test_2c, test_df['ret_next'].values, test_df['session_key'].values, thr_hi=thr_hi, thr_lo=thr_lo)

# AUC для 2-class на подмножестве BUY/SELL (для справедливого сравнения метрик)
mask_binary = (y_val != 1)  # SELL or BUY
y_val_bin = (y_val == 2).astype(int)[mask_binary]
y_test_bin = (y_test == 2).astype(int)[(y_test != 1)]
proba_val_bin = proba_val_2c[mask_binary]
proba_test_bin = proba_test_2c[(y_test != 1)]
auc_val_2c = roc_auc_score(y_val_bin, proba_val_bin) if len(np.unique(y_val_bin)) > 1 else np.nan
auc_test_2c = roc_auc_score(y_test_bin, proba_test_bin) if len(np.unique(y_test_bin)) > 1 else np.nan

print(f'=== 2-Class Backtest (prod_cat_seq, full data, 25-70 thr_hi={thr_hi} thr_lo={thr_lo}) ===')
print(f'VAL:  net={bt_val_2c["net_%"]:+.2f}%  trades={bt_val_2c["trades"]}  avg/trade={bt_val_2c["avg_%_per_trade"]:.4f}%')
print(f'TEST: net={bt_test_2c["net_%"]:+.2f}%  trades={bt_test_2c["trades"]}  avg/trade={bt_test_2c["avg_%_per_trade"]:.4f}%')
print(f'AUC val (BUY/SELL only): {auc_val_2c:.4f} | test: {auc_test_2c:.4f}')
print()
print('=== Comparison Table ===')
print(f'{"Metric":<25} {"2-Class":>15} {"3-Class":>15}')
print('-' * 55)
print(f'{"Accuracy test":<25} {"N/A":>15} {acc_test:>15.4f}')
print(f'{"AUC test (BUY/SELL)":<25} {auc_test_2c:>15.4f} {"N/A":>15}')
print(f'{"Val avg_%_per_trade":<25} {bt_val_2c["avg_%_per_trade"]:>15.4f} {bt_val_3c["avg_%_per_trade"]:>15.4f}')
print(f'{"Test avg_%_per_trade":<25} {bt_test_2c["avg_%_per_trade"]:>15.4f} {bt_test_3c["avg_%_per_trade"]:>15.4f}')
print(f'{"Val net_%":<25} {bt_val_2c["net_%"]:>15.2f} {bt_val_3c["net_%"]:>15.2f}')
print(f'{"Test net_%":<25} {bt_test_2c["net_%"]:>15.2f} {bt_test_3c["net_%"]:>15.2f}')

=== 2-Class Backtest (prod_cat_seq, full data, 25-70 thr_hi=0.7 thr_lo=0.25) ===
VAL:  net=+559.00%  trades=314  avg/trade=1.7802%
TEST: net=+1475.60%  trades=768  avg/trade=1.9213%
AUC val (BUY/SELL only): 0.7419 | test: 0.7136

=== Comparison Table ===
Metric                            2-Class         3-Class
-------------------------------------------------------
Accuracy test                         N/A          0.5238
AUC test (BUY/SELL)                0.7136             N/A
Val avg_%_per_trade                1.7802          0.5412
Test avg_%_per_trade               1.9213          0.3543
Val net_%                          559.00         2381.79
Test net_%                        1475.60         2805.27


## 6. Сохранение 3-class модели (опционально)

In [6]:
bundle_3c = {
    'model': model_3c,
    'model_name': 'LightGBM_3class_seq_5_15_30_60',
    'scaler': scaler,
    'features': ALL_FEATURES,
    'base_features': BASE_FEATURES,
    'seq_windows': SEQ_WINDOWS,
    'seq_key_feats': KEY_FEATS,
    'target': TARGET_COL,
    'num_class': 3,
    'class_map': {-1: 0, 0: 1, 1: 2},
    'commission_rt': COMMISSION_RT,
    'acc_val': acc_val,
    'acc_test': acc_test,
    'backtest_val': bt_val_3c,
    'backtest_test': bt_test_3c,
    'split': {'train': f'{min(train_dates)}..{max(train_dates)}', 'val': str(dates[7]), 'test': f'{dates[8]}, {dates[9]}'},
}

os.makedirs(os.path.dirname(MODEL_3CLASS_PATH), exist_ok=True)
joblib.dump(bundle_3c, MODEL_3CLASS_PATH, compress=3)
print(f'Saved: {MODEL_3CLASS_PATH} ({os.path.getsize(MODEL_3CLASS_PATH)/1024/1024:.1f} MB)')

Saved: c:\project\trading_bot_2Engine\models\prod_lgbm_3class.joblib (1.4 MB)
